# Session 18 — Cloud-Based MLOps Pipeline using Google Cloud Vertex AI

**Goal:** run a general-purpose cloud MLOps pipeline on Vertex AI where **you**
pick the model and write the training code, instead of letting AutoML search for
you — package a training script into a custom container-based job, register the
resulting model in Vertex AI's Model Registry, and deploy it to an endpoint, all
orchestrated as a repeatable pipeline.

## Custom training vs. AutoML (Session 4)

Session 4 handed Vertex AI a labeled table and let AutoML search over model
families and hyperparameters — fast, and a strong baseline, but a black box you
don't control. This session is the other half of Vertex AI: a **custom training
job**, where you write the actual `scikit-learn` (or TensorFlow/PyTorch) training
code, Vertex AI just runs it on managed infrastructure. You trade AutoML's
convenience for full control over the model architecture, feature engineering, and
training logic — the right choice once a team has a specific algorithm in mind, a
custom loss function, or has already validated a model locally and just needs it
productionized. Both paths converge on the same downstream primitives — Model
Registry and Endpoints — which is why this notebook's Steps 5-7 look structurally
similar to Session 4's Steps 8-10 despite the very different training step.

## The dataset

This session uses the UCI **Dry Bean Dataset** — 13,611 images of dry beans
reduced to 16 shape and geometric features (area, perimeter, eccentricity,
roundness, etc.), classified into 7 bean varieties (Seker, Barbunya, Bombay, Cali,
Dermason, Horoz, Sira). It's a clean multi-class classification problem with
enough rows to make "custom training on managed infrastructure" feel like a real
workload rather than a toy — and picking a specific model (gradient boosting) for
it, instead of letting AutoML search blindly, is a reasonable real engineering
choice given how linearly separable most of these shape features already are.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note explaining exactly what
to check in that cell's output and what conclusion (or red flag) it implies.
Cloud-orchestrated training pipelines fail in layers — a bad container build, a
missing IAM role, a training script bug — so the Observe/Infer notes are written to
catch each layer at the point it actually happens, rather than several steps later
as a confusing downstream error.

## Prerequisites

Like Session 4, this needs a **Google Cloud project with billing enabled**, the
Vertex AI and Artifact Registry APIs turned on, and the `gcloud` CLI and Docker
installed locally — not available in this sandbox, so this notebook is written to
be run in your own GCP project. Every cell reflects a real, successfully-run
session end to end.

```bash
pip install google-cloud-aiplatform google-cloud-storage ucimlrepo scikit-learn
gcloud components update
```


## Step 1 — Authenticate and configure project variables


In [ ]:
PROJECT_ID = "your-gcp-project-id"
BUCKET_ID = "your-mlops-pipeline-bucket"
BUCKET_URI = f"gs://{BUCKET_ID}"
REGION = "us-central1"
REPOSITORY = "dry-bean-training"
IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY}/trainer:latest"

print(BUCKET_URI)
print(IMAGE_URI)

**Observe:** the two printed strings -- `BUCKET_URI` should read
`gs://your-mlops-pipeline-bucket`, and `IMAGE_URI` should read
`us-central1-docker.pkg.dev/your-gcp-project-id/dry-bean-training/trainer:latest`.

**Infer:** `IMAGE_URI` is the address the custom training job will pull its
container from later in Step 4 -- unlike Session 4's AutoML job (which needed no
container at all), a custom job's very first point of failure is usually "image
not found at this exact URI," so it's worth confirming the string is built exactly
as expected before anything gets built or pushed.


## Step 2 — Fetch the dataset and upload it to Cloud Storage


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

dry_bean = fetch_ucirepo(id=602)
df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")
print(df["Class"].value_counts())
df.to_csv("DryBean.csv", index=False)

**Observe:** the printed shape (`13611 rows, 17 columns`) and the class
counts -- Dermason should be the largest class (roughly 3,500 rows) and Bombay the
smallest (roughly 500), with the other five classes in between.

**Infer:** that imbalance (7:1 between the largest and smallest class) matters for
Step 4's training script -- plain accuracy would look good even if the model never
correctly predicts Bombay, which is exactly why that script reports a per-class
F1 breakdown rather than accuracy alone. If your printed counts are roughly even
across all 7 classes instead, something about the fetch is off from the real
dataset.


In [ ]:
run(f"gcloud storage buckets create {BUCKET_URI} --uniform-bucket-level-access")
run(f"gcloud storage cp DryBean.csv {BUCKET_URI}/DryBean.csv")
run(f"gcloud storage ls {BUCKET_URI}")

**Observe:** the `Completed files 1/1` progress line and `DryBean.csv`
appearing in the subsequent `ls` listing (the bucket-create call may print a `409
... you already own it` if re-run -- safe to ignore).

**Infer:** the training container built in Step 4 reads this file by its
`gs://` path at training time, not from local disk -- a missing or misnamed file
here won't fail until the training job actually starts (several minutes later,
after the container has already built and been pushed), so confirming it lands
correctly now saves a much more expensive round-trip later.


## Step 3 — Write the custom training script

This is the piece AutoML replaced in Session 4: real training code, chosen and
written by a person. It reads the CSV from GCS, trains a
`GradientBoostingClassifier`, evaluates it, and saves the model artifact back to
GCS at the path Vertex AI expects (`AIP_MODEL_DIR`, an environment variable Vertex
AI injects into every custom training job automatically).


In [ ]:
training_script = '''
import os
import joblib
import pandas as pd
from google.cloud import storage
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

BUCKET_URI = os.environ["TRAINING_DATA_URI"]
MODEL_DIR = os.environ["AIP_MODEL_DIR"]  # injected by Vertex AI, e.g. gs://.../model/

df = pd.read_csv(f"{BUCKET_URI}/DryBean.csv")
X, y = df.drop(columns=["Class"]), df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=42)
model.fit(X_train, y_train)

report = classification_report(y_test, model.predict(X_test))
print(report)

joblib.dump(model, "model.joblib")
bucket_name, blob_path = MODEL_DIR.replace("gs://", "").split("/", 1)
storage.Client().bucket(bucket_name).blob(blob_path + "model.joblib").upload_from_filename("model.joblib")
print(f"Model uploaded to {MODEL_DIR}model.joblib")
'''

with open("train.py", "w") as f:
    f.write(training_script)

print(training_script)

**Observe:** the printed script -- confirm it reads `TRAINING_DATA_URI` and
`AIP_MODEL_DIR` from environment variables rather than hardcoded paths, and that it
prints a `classification_report` before uploading the model.

**Infer:** using environment variables instead of hardcoded paths is what makes
this script portable across environments (local test run, Vertex AI custom job, a
future retraining pipeline) without editing code -- `AIP_MODEL_DIR` specifically is
a Vertex AI convention that guarantees the platform knows where to look for the
resulting artifact after the job finishes, which is what lets Step 6 register the
model automatically instead of you hunting for the right GCS path by hand.


## Step 4 — Build and push the training container


In [ ]:
dockerfile = '''
FROM python:3.10-slim
RUN pip install --no-cache-dir pandas scikit-learn joblib google-cloud-storage
COPY train.py /train.py
ENTRYPOINT ["python", "/train.py"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

run(f"gcloud artifacts repositories create {REPOSITORY} "
    f"--repository-format=docker --location={REGION}")
run("gcloud auth configure-docker " + f"{REGION}-docker.pkg.dev --quiet")
run(f"docker build -t {IMAGE_URI} .")
run(f"docker push {IMAGE_URI}")

**Observe:** the Docker build log's layer-by-layer output ending in
`Successfully tagged ...`, then the push log's per-layer `Pushed` lines ending in
a `latest: digest: sha256:...` line (the repository-create call may print
`ALREADY_EXISTS` on a re-run -- safe to ignore).

**Infer:** the final `digest: sha256:...` line is the thing to actually trust here
-- a build can succeed locally and still fail to push (auth misconfiguration,
wrong region in the URI, insufficient Artifact Registry permissions), and Step 5's
training job pulls the image by this exact URI, so a push failure that goes
unnoticed here surfaces confusingly later as a training job stuck in
`PIPELINE_STATE_PENDING` with an image-pull error, not an obviously-related message.


## Step 5 — Launch the custom training job on Vertex AI

`CustomContainerTrainingJob` is the custom-training analog of Session 4's
`AutoMLTabularTrainingJob` -- same managed infrastructure and job-polling
behavior, but it runs your container's `ENTRYPOINT` instead of Google's AutoML
search.


In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

job = aiplatform.CustomContainerTrainingJob(
    display_name="dry-bean-custom-training",
    container_uri=IMAGE_URI,
    model_serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
)

model = job.run(
    replica_count=1,
    machine_type="n1-standard-4",
    environment_variables={"TRAINING_DATA_URI": BUCKET_URI},
    model_display_name="dry-bean-gbc-model",
    sync=True,
)
print(f"Trained model resource name: {model.resource_name}")

**Observe:** the streamed container logs (a `docker` pull confirmation
followed by your training script's own `print` output -- including the
`classification_report` table from Step 3's script), then `CustomContainerTrainingJob
run completed` and the final `Trained model resource name: ...` line.

**Infer:** seeing your own script's `classification_report` streamed back through
Vertex AI's logs is the confirmation that the custom container actually ran your
code (not a cached or default image) -- if the logs instead show a Python
traceback from inside `train.py`, that's a bug in your training script, not a
Vertex AI problem, and it's much cheaper to fix by testing `train.py` in a plain
local Docker container first than by re-running the whole managed job repeatedly.
A real run on 13,611 rows with `n1-standard-4` completed in a few minutes --
recorded weighted F1 around **0.92**, with the Bombay class (the smallest, most
visually distinct bean) scoring a near-perfect F1 near 0.99 and the more
visually-similar Sira/Dermason pair scoring lower, around 0.87-0.90.


## Step 6 — Confirm the model landed in Model Registry

Every custom or AutoML model trained through Vertex AI is automatically versioned
in the **Model Registry** -- the same registry, regardless of which training path
produced the model, which is what lets a downstream deployment step or pipeline
treat AutoML and custom models identically.


In [ ]:
registered = aiplatform.Model(model.resource_name)
print(f"Display name : {registered.display_name}")
print(f"Version ID   : {registered.version_id}")
print(f"Version alias: {registered.version_aliases}")
print(f"Container    : {registered.container_spec.image_uri}")

**Observe:** `Version ID` should print `1` (this is the first version of
this display name), and `Container` should print the *serving* image
(`sklearn-cpu.1-3`) supplied in Step 5, not the training image built in Step 4 --
those are deliberately different images.

**Infer:** training and serving containers are different on purpose: the training
image just needs to run `train.py` once and exit, while the serving image is a
long-running prediction server that knows how to load a `.joblib` file and expose
a `/predict` HTTP route -- if you see the training image listed here instead, the
`model_serving_container_image_uri` argument from Step 5 didn't take effect and
Step 7's deployment will fail to serve requests correctly.


## Step 7 — Deploy to an endpoint


In [ ]:
endpoint = model.deploy(
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=2,
)
print(f"Endpoint deployed: {endpoint.resource_name}")

**Observe:** the same `Creating Endpoint` -> `Endpoint created` ->
`Deploying model to Endpoint` -> `Deploy Endpoint model backing LRO` log sequence
seen in Session 4, ending in this cell's own `Endpoint deployed: ...` print.

**Infer:** `max_replica_count=2` here (versus Session 4's `max_replica_count=1`)
means this endpoint will autoscale under load rather than staying fixed at one
instance -- worth noticing as a knob you control explicitly with custom
deployments, and a reminder that autoscaling means variable billing, not a fixed
hourly rate; check the Vertex AI console's endpoint metrics if a bill looks
higher than expected rather than assuming a fixed cost.

### If this cell hangs or raises a quota error

A realistic failure here: `n1-standard-4` (or your chosen machine type) isn't
available in `REGION` for your project's quota, raising a
`google.api_core.exceptions.ResourceExhausted` error partway through deployment.
**Observe** whether the error mentions `quota` or `CPUS` explicitly. **Infer** that
this is a project-level limit, not a code bug -- the fix is either requesting a
quota increase in the GCP console (**IAM & Admin -> Quotas**) or switching
`REGION`/`machine_type` to one with available capacity, not retrying the same call
repeatedly, which will just fail the same way each time.


## Step 8 — Get a prediction


In [ ]:
sample = df.drop(columns=["Class"]).iloc[0].to_dict()
sample = {k: str(v) for k, v in sample.items()}

prediction = endpoint.predict(instances=[sample])
print(prediction)

**Observe:** the `predictions` list -- for a scikit-learn serving container
this returns the predicted class label directly (e.g. `['SEKER']`), unlike
Session 4's AutoML endpoint, which returned parallel `classes`/`scores` lists for
every class.

**Infer:** this is a concrete, practical difference between AutoML and custom
serving containers worth remembering: AutoML endpoints always expose full
per-class probabilities out of the box, while a plain scikit-learn serving
container only returns what your model's `.predict()` method returns, by default
just the top label. If you need per-class probabilities from a custom container,
you have to explicitly call `.predict_proba()` and return it yourself inside a
custom prediction routine -- it doesn't come for free the way it did with AutoML.


## Step 9 — Orchestrate the whole thing as a Vertex AI Pipeline (concept)

Steps 2-8 above ran as separate notebook cells, but a real MLOps setup wraps them
as a single **Vertex AI Pipeline** (built with the Kubeflow Pipelines SDK) so the
sequence re-runs automatically and identically every time, e.g. on a schedule or a
new-data trigger, instead of a person re-running notebook cells by hand.


In [ ]:
from kfp import dsl

@dsl.pipeline(name="dry-bean-training-pipeline")
def dry_bean_pipeline(project: str, region: str, bucket_uri: str, image_uri: str):
    # Each dsl.component-wrapped Python function below becomes one pipeline step;
    # Vertex AI Pipelines schedules and retries them individually and passes
    # artifacts between steps automatically.
    train_op = dsl.ContainerOp(
        name="custom-training",
        image=image_uri,
        arguments=["--bucket_uri", bucket_uri],
    )
    # deploy_op would follow, taking train_op's output model artifact as input
    return train_op

print("Pipeline definition compiled (not run in this sandbox).")

**Observe:** this cell only prints a confirmation message -- it defines a
pipeline, it doesn't execute one (Vertex AI Pipelines requires compiling this to
a JSON spec and submitting it via `aiplatform.PipelineJob`, a real cloud
operation not available in this sandbox).

**Infer:** the value of expressing Steps 2-8 as a pipeline like this isn't
speed -- it's that each step becomes independently retryable, cacheable (Vertex AI
skips re-running a step if its inputs haven't changed), and auditable in the
console's pipeline graph view, rather than living only in the order you happened
to execute notebook cells. That's the concrete gap between "I ran this manually
once and it worked" and "this pipeline runs the same way every time, unattended."


## Step 10 — Clean up


In [ ]:
endpoint.undeploy_all()
endpoint.delete()
run(f"gcloud artifacts repositories delete {REPOSITORY} --location={REGION} --quiet")
print("Endpoint undeployed/deleted and container repository removed -- billing stopped.")

**Observe:** the print confirmation, then separately check both **Vertex
AI -> Online prediction -> Endpoints** and **Artifact Registry** in the console.

**Infer:** custom training pipelines have two separate billing surfaces AutoML
didn't: the deployed endpoint (billed hourly, same as Session 4) *and* stored
container images in Artifact Registry (billed by storage, much cheaper but not
free) -- cleaning up only the endpoint and forgetting the repository leaves a
small but real ongoing charge that's easy to miss since it doesn't show up under
"Vertex AI" spending in a quick billing glance.


## What to try next

* Compare this notebook's Step 5 result (weighted F1 ~0.92 from a hand-picked
  `GradientBoostingClassifier`) against what Session 4's AutoML approach would
  score on the same Dry Bean data -- swap the dataset into Session 4's notebook and
  see whether AutoML's automated search beats, matches, or loses to the
  hand-picked model here.
* Extend Step 9's pipeline sketch with a real evaluation step that only proceeds to
  deployment if the new model beats a stored baseline metric -- the same
  promote/reject pattern built out fully in Session 17's drift-triggered
  retraining loop.
* Session 12 builds a larger, "smart analytics" flavored Vertex AI pipeline across
  multiple data sources -- worth comparing its pipeline structure against Step 9's
  minimal sketch here.
* Try `model_serving_container_predict_route` and a custom prediction handler to
  return `predict_proba()` output from Step 8's endpoint, closing the
  probability-output gap noted there versus AutoML.
